In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


print("=== LOADING FILES ===")
data_path = Path("../data/")   # full path to the HEF folder

train = pd.read_csv(data_path / "mimic_train_HEF.csv", low_memory=False)
test  = pd.read_csv(data_path / "mimic_test_HEF.csv",  low_memory=False)

print("\n=== SHAPES ===")
print("TRAIN:", train.shape)
print("TEST:", test.shape)

# -------------------------------
# BASIC COLUMN + DTYPE INFO
# -------------------------------
print("\n=== TRAIN COLUMNS & DTYPES ===")
print(train.dtypes)

print("\n=== TEST COLUMNS & DTYPES ===")
print(test.dtypes)

# -------------------------------
# MISSINGNESS SUMMARY
# -------------------------------
print("\n=== MISSINGNESS (TRAIN) ===")
missing_train = train.isna().mean().sort_values(ascending=False)
print(missing_train)

print("\n=== MISSINGNESS (TEST) ===")
missing_test = test.isna().mean().sort_values(ascending=False)
print(missing_test)

# -------------------------------
# COLUMNS THAT EXIST ONLY IN TRAIN (LIKELY TARGET/LEAKAGE)
# -------------------------------
print("\n=== COLUMNS IN TRAIN BUT NOT IN TEST ===")
train_only = list(set(train.columns) - set(test.columns))
print(train_only)

print("\n=== COLUMNS IN TEST BUT NOT IN TRAIN ===")
test_only = list(set(test.columns) - set(train.columns))
print(test_only)

# -------------------------------
# IDENTIFY ID-LIKE COLUMNS
# -------------------------------
print("\n=== ID-LIKE COLUMNS DETECTED ===")
id_cols = [c for c in train.columns if "ID" in c.upper() or "ID" in c.lower()]
print(id_cols)

# -------------------------------
# NUMERIC SUMMARY
# -------------------------------
print("\n=== NUMERIC SUMMARY (TRAIN) ===")
num_summary = train.describe().transpose()
print(num_summary)

# -------------------------------
# CATEGORICAL SUMMARY
# -------------------------------
print("\n=== CATEGORICAL COLUMNS (TRAIN) ===")
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print(cat_cols)

print("\n=== CARDINALITY OF CATEGORICAL COLUMNS ===")
cat_cardinality = {col: train[col].nunique(dropna=True) for col in cat_cols}
for col, card in cat_cardinality.items():
    print(f"{col}: {card}")

# -------------------------------
# CORRELATION WITH TARGETS (if present)
# -------------------------------
targets = [col for col in ["HOSPITAL_EXPIRE_FLAG", "LOS"] if col in train.columns]

if targets:
    print("\n=== CORRELATIONS WITH TARGETS ===")
    for target in targets:
        print(f"\n--- Correlation with {target} ---")
        try:
            corr = train.corr(numeric_only=True)[target].sort_values(ascending=False)
            print(corr)
        except:
            print(f"Could not compute correlation for {target}")
else:
    print("\n=== No target columns detected (unexpected for train.csv) ===")

# -------------------------------
# UNIQUE VALUES FOR TARGETS
# -------------------------------
if "HOSPITAL_EXPIRE_FLAG" in train.columns:
    print("\n=== VALUE COUNTS: HOSPITAL_EXPIRE_FLAG ===")
    print(train["HOSPITAL_EXPIRE_FLAG"].value_counts(dropna=False))

if "LOS" in train.columns:
    print("\n=== LOS BASIC STATS ===")
    print(train["LOS"].describe())


=== LOADING FILES ===

=== SHAPES ===
TRAIN: (20885, 44)
TEST: (5221, 39)

=== TRAIN COLUMNS & DTYPES ===
HOSPITAL_EXPIRE_FLAG      int64
subject_id                int64
hadm_id                   int64
icustay_id                int64
HeartRate_Min           float64
HeartRate_Max           float64
HeartRate_Mean          float64
SysBP_Min               float64
SysBP_Max               float64
SysBP_Mean              float64
DiasBP_Min              float64
DiasBP_Max              float64
DiasBP_Mean             float64
MeanBP_Min              float64
MeanBP_Max              float64
MeanBP_Mean             float64
RespRate_Min            float64
RespRate_Max            float64
RespRate_Mean           float64
TempC_Min               float64
TempC_Max               float64
TempC_Mean              float64
SpO2_Min                float64
SpO2_Max                float64
SpO2_Mean               float64
Glucose_Min             float64
Glucose_Max             float64
Glucose_Mean            float6